In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# plot
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")


In [ ]:
# Task1 : drop order id
df.columns.drop("Order_ID")

In [ ]:
# Task 2: Write your code here: missing values
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# drop mising values :

df.dropna(subset=['Weather'])
df.dropna(subset=['Delivery_Time']) # target value
df.dropna(subset=['Traffic_Level'])
df.dropna(subset=['Courier_Experience_yrs'])
df.dropna(subset=['Time_of_Day'])



In [ ]:
# Task 3: Write your code here: remove duplicate

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder , OneHotEncoder

In [ ]:
# Task 4: Write your code here: encode categorical : Weather	Traffic_Level	Time_of_Day	Vehicle_Type

# first find them

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day',  'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', df) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
df = standard_scaler.fit_transform(df) # Apply fit_transform

print('\nData after scaling:\n', df) #show after scaling


In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 5: Write your code here:
# plot
df["Delivery_Time"].hist(bins=30, edgecolor='black')

plt.xlabel("Delivery_Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()



In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Task 1: Write your code here:
# first define the x , y

# Define features (X) and target (y)
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
X = df[feature_cols]
y = df['Delivery_Time']

#split into train and test
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



In [ ]:
# Task 2,3,4,5: Write your code here:

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

#evlauate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)


print(f"MAE:  ${mae:,.2f}")

#print



In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: